# 本地调试 - 布局生成
本文档展示了如何在本地调试并生成布局和样式。包含了 Markdown 解析、布局模型推理、样式生成、HTML 输出等过程。

In [12]:
import json
import os
import torch
import sys
from collections import namedtuple

# 路径设置 - 适配 Jupyter Notebook
current_dir = os.getcwd()  # 使用 os.getcwd() 获取当前工作目录
src_dir = os.path.abspath(os.path.join(current_dir, "../../../"))
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# 引入相关模块
from src.api.schemas.layout import LayoutRequest, LayoutResponse
from src.services.layout_agent.models.layout_model import DynamicPPONetwork
from src.services.layout_agent.utils.layout_generator import generate_layout
from src.utils.markdown_parser import MarkdownParser
from src.utils.block_generator import BlockGenerator
from src.services.style_agent.rule_generator import StyleRuleGenerator
from src.services.style_agent.runtime.runtime_manager import RuntimeManager
from src.utils.config import STYLE_CONFIG

# 定义 Block
Block = namedtuple('Block', ['id', 'content_type', 'content_length', 'min_width', 'min_height'])

# 全局变量
layout_model = None
markdown_parser = None

# 路径设置
current_dir = os.getcwd()  # Jupyter Notebook 环境下使用当前工作目录
layout_model_path = os.path.join(current_dir, "data/models/layout/layoutModel_05.pth")


In [13]:
def init_components():
    """初始化布局模型和Markdown解析器"""
    global layout_model, markdown_parser

    # 初始化布局模型
    if layout_model is None:
        layout_model = DynamicPPONetwork()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        layout_model.load_state_dict(torch.load(layout_model_path, map_location=device, weights_only=True))
        layout_model.eval()

    # 初始化Markdown解析器
    if markdown_parser is None:
        markdown_parser = MarkdownParser()

In [17]:
def generate_layout_local(request: LayoutRequest):
    """本地生成布局和样式"""
    try:
        # 初始化
        init_components()

        # 输入卡片宽高和内容
        card_width = int(request.card_width) if request.card_width else 1200
        card_height = int(request.card_height) if request.card_height else 800

        # 使用 Markdown 解析器解析输入
        json_content = markdown_parser.parse_to_json(request.markdown_text)
        generator = BlockGenerator({'width': card_width, 'height': card_height}, STYLE_CONFIG)
        layout_infos, blocks = generator.generate_blocks(json_content)

        # 使用模型生成布局
        final_positions = generate_layout(
            layout_model,
            card_width,
            card_height,
            blocks
        )

        # 更新布局并生成 JSON
        layout_json = generator.update_layout_and_get_content(json_content, layout_infos, final_positions, card_width)



        # 样式优化
        card_size = {"width": card_width, "height": card_height}
        style_generator = StyleRuleGenerator(
            layout_info=layout_json,
            card_size=card_size,
            theme_color=request.theme_color
        )
        style_rules = style_generator.generate()

        # 运行时管理器生成 HTML
        runtime = RuntimeManager(
            layout_info=layout_json,
            card_size=card_size,
            style_rules=style_rules
        )
        html = runtime.generate_html()

        # 保存 HTML 到本地文件
        output_file_path =  "generated.html"
        with open(output_file_path, "w", encoding="utf-8") as f:
            f.write(html)

        print("HTML 文件已生成并保存至:", output_file_path)
        return LayoutResponse(layout_json=html)

    except Exception as e:
        print(f"生成布局时出错: {str(e)}")
        return None

In [ ]:
# 本地调用示例
markdown_input = """
# 请调休假方案

## 节假日信息
根据中国国家假日办发布的节假日放假信息，下一个距离今天最近的节假日为**双休日**，预计放假天数为**2天**。

## 请假方案
请假开始时间：**2024-11-27**
请假结束时间：**2024-11-31**
请假时长：**5天**

## 推荐旅游目的地
1. **北京** - 距离：**0公里**
   - 推荐景点：**故宫、天安门、颐和园**
"""

# 构建输入请求
request = LayoutRequest(
    markdown_text=markdown_input,
    card_width="1200",
    card_height="800",
    theme_color=""
)

# 生成布局
response = generate_layout_local(request)

if response:
    print("生成的布局 HTML 内容：")
else:
    print("布局生成失败。")

HTML 文件已生成并保存至: generated.html
生成的布局 HTML 内容：


In [21]:
import json
import os
import torch
import sys
from collections import namedtuple

# 路径设置 - 适配 Jupyter Notebook
current_dir = os.getcwd()  # 使用 os.getcwd() 获取当前工作目录
src_dir = os.path.abspath(os.path.join(current_dir, "../../../"))
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# 引入相关模块
from src.api.schemas.layout import LayoutRequest, LayoutResponse
from src.services.layout_agent.models.layout_model import DynamicPPONetwork
from src.services.layout_agent.utils.layout_generator import generate_layout
from src.utils.markdown_parser import MarkdownParser
from src.utils.block_generator import BlockGenerator
from src.services.style_agent.rule_generator import StyleRuleGenerator
from src.services.style_agent.runtime.runtime_manager import RuntimeManager
from src.utils.config import STYLE_CONFIG

# 定义 Block
Block = namedtuple('Block', ['id', 'content_type', 'content_length', 'min_width', 'min_height'])

# 全局变量
layout_model = None
markdown_parser = None

# 路径设置
current_dir = os.getcwd()  # Jupyter Notebook 环境下使用当前工作目录
layout_model_path = os.path.join(current_dir, "data/models/layout/layoutModel_05.pth")

# 初始化组件
def init_components():
    """初始化布局模型和Markdown解析器"""
    global layout_model, markdown_parser

    # 初始化布局模型
    if layout_model is None:
        layout_model = DynamicPPONetwork()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        layout_model.load_state_dict(torch.load(layout_model_path, map_location=device, weights_only=True))
        layout_model.eval()
        print("布局模型已加载。")

    # 初始化Markdown解析器
    if markdown_parser is None:
        markdown_parser = MarkdownParser()
        print("Markdown解析器已初始化。")

# 解析 Markdown 文本
def parse_markdown(markdown_text: str):
    """解析Markdown文本为JSON"""
    global markdown_parser
    json_content = markdown_parser.parse_to_json(markdown_text)
    print("Markdown解析结果:")
    display(json_content)
    return json_content

# 生成布局块
def generate_blocks(json_content, card_width, card_height):
    """根据JSON内容生成布局块"""
    generator = BlockGenerator({'width': card_width, 'height': card_height}, STYLE_CONFIG)
    layout_infos, blocks = generator.generate_blocks(json_content)
    print("生成的布局块:")
    display(blocks)
    return layout_infos, blocks

# 使用模型生成布局
def generate_layout_positions(layout_model, card_width, card_height, blocks):
    """使用模型生成布局位置"""
    final_positions = generate_layout(layout_model, card_width, card_height, blocks)
    print("布局模型输出的位置:")
    display(final_positions)
    return final_positions

# 更新布局并生成 HTML
def generate_layout_html(json_content, layout_infos, final_positions, card_width, card_height, theme_color):
    """更新布局并生成 HTML"""
    generator = BlockGenerator({'width': card_width, 'height': card_height}, STYLE_CONFIG)
    layout_json = generator.update_layout_and_get_content(json_content, layout_infos, final_positions, card_width)
    print("更新后的布局 JSON:")
    display(json.dumps(layout_json, ensure_ascii=False, indent=4))

    # 样式优化
    style_generator = StyleRuleGenerator(
        layout_info=layout_json,
        card_size={"width": card_width, "height": card_height},
        theme_color=theme_color
    )
    style_rules = style_generator.generate()
    print("生成的样式规则:")
    display(style_rules)

    # 运行时管理器生成 HTML
    runtime = RuntimeManager(
        layout_info=layout_json,
        card_size={"width": card_width, "height": card_height},
        style_rules=style_rules
    )
    html = runtime.generate_html()
    print("生成的 HTML 内容:")
    display(html)
    return html

# 主流程
if __name__ == "__main__":
    init_components()

    # 示例 Markdown 文本
    markdown_text = """
    # 标题1
    ## 子标题1.1
    这是一个段落。
    - 列表项1
    - 列表项2
    ## 子标题1.2
    这是另一个段落。
    """

    # 配置卡片宽高
    card_width = 1200
    card_height = 800
    theme_color = "#FF5733"  # 可选主题颜色

    # 分步骤处理
    json_content = parse_markdown(markdown_text)
    layout_infos, blocks = generate_blocks(json_content, card_width, card_height)
    final_positions = generate_layout_positions(layout_model, card_width, card_height, blocks)
    html = generate_layout_html(json_content, layout_infos, final_positions, card_width, card_height, theme_color)


FileNotFoundError: [Errno 2] No such file or directory: '/home/liyue/dingdaocode/aigui-model-service/src/api/endpoints/data/models/layout/layoutModel_05.pth'